In [29]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import (
    LLMChainExtractor,
    EmbeddingsFilter,
    DocumentCompressorPipeline
)

from dotenv import load_dotenv
load_dotenv()

True

In [5]:
embedding = OpenAIEmbeddings(model = "text-embedding-3-small")
llm = ChatOpenAI(model = 'gpt-4o-mini', temperature = 0.1)

In [6]:
# Dummy documents covering different topics, each with a mix of relevant and tangential info
docs = [
    Document(
        page_content=(
            "Artificial intelligence has made remarkable strides in natural language processing, "
            "with large language models now capable of generating human-quality text and code. "
            "Computer vision systems can identify objects in images with superhuman accuracy, "
            "powering applications from autonomous vehicles to medical imaging diagnostics. "
            "However, the rapid advancement of AI has raised significant ethical concerns about "
            "job displacement, algorithmic bias, and the concentration of power among a few tech companies."
        ),
        metadata={"topic": "artificial_intelligence"},
    ),
    Document(
        page_content=(
            "Global temperatures have risen by approximately 1.1 degrees Celsius since pre-industrial "
            "times, driven primarily by the burning of fossil fuels. The melting of polar ice caps has "
            "accelerated, contributing to rising sea levels that threaten coastal communities worldwide. "
            "Renewable energy adoption is growing rapidly, with solar and wind power becoming cheaper "
            "than coal in many regions. Governments are implementing carbon pricing mechanisms and "
            "investing in green infrastructure to meet Paris Agreement targets."
        ),
        metadata={"topic": "climate_change"},
    ),
    Document(
        page_content=(
            "NASA's Artemis program aims to return humans to the Moon by the mid-2020s, establishing "
            "a sustainable presence as a stepping stone to Mars. Private companies like SpaceX are "
            "developing reusable rocket technology that has dramatically reduced launch costs. "
            "The James Webb Space Telescope has captured unprecedented images of distant galaxies, "
            "revealing new insights about the early universe. Asteroid mining is being explored as a "
            "potential source of rare minerals needed for electronics manufacturing."
        ),
        metadata={"topic": "space_exploration"},
    ),
    Document(
        page_content=(
            "CRISPR gene editing technology has revolutionized medical genomics, enabling precise "
            "modifications to DNA sequences that were previously impossible. Researchers are using "
            "genomic data to develop personalized medicine approaches, tailoring treatments based on "
            "an individual's genetic profile. Recent breakthroughs in mRNA technology, accelerated by "
            "COVID-19 vaccine development, are now being applied to cancer immunotherapy and rare "
            "genetic disorders. Hospital information systems are increasingly integrating genomic data "
            "to support clinical decision-making at the point of care."
        ),
        metadata={"topic": "medicine"},
    ),
    Document(
        page_content=(
            "The global economy is navigating a period of high inflation driven by supply chain "
            "disruptions, energy price volatility, and post-pandemic demand surges. Central banks "
            "worldwide have raised interest rates aggressively to combat inflation, impacting housing "
            "markets and consumer spending. Cryptocurrency regulation is becoming a priority for "
            "financial authorities, with the EU's MiCA framework setting a global precedent. "
            "Trade tensions between major economies continue to reshape global supply chains, "
            "pushing companies toward nearshoring and diversification strategies."
        ),
        metadata={"topic": "economics"},
    ),
    Document(
        page_content=(
            "Quantum computing has reached a critical milestone with several companies demonstrating "
            "quantum advantage on specific computational tasks. Error correction remains the biggest "
            "challenge, as current quantum processors are highly susceptible to noise and decoherence. "
            "Quantum simulation of molecular structures could transform drug discovery by accurately "
            "modeling protein folding and chemical interactions. Major tech companies and governments "
            "are investing billions in quantum research, viewing it as essential for national security "
            "and economic competitiveness."
        ),
        metadata={"topic": "quantum_computing"},
    ),
]

print(f"Created {len(docs)} documents")

Created 6 documents


In [8]:
docs[0].page_content

'Artificial intelligence has made remarkable strides in natural language processing, with large language models now capable of generating human-quality text and code. Computer vision systems can identify objects in images with superhuman accuracy, powering applications from autonomous vehicles to medical imaging diagnostics. However, the rapid advancement of AI has raised significant ethical concerns about job displacement, algorithmic bias, and the concentration of power among a few tech companies.'

In [9]:
vectorstore = InMemoryVectorStore.from_documents(documents= docs, embedding= embedding)
base_retriever = vectorstore.as_retriever(search_kwargs = {'k':3})

In [21]:
query = "How is CRISPR acting as a big enabler in creating personalized medicine?"

base_result = base_retriever.invoke(query)

print(f"Query: {query}")
for i, doc in enumerate(base_result,1):
    print(f"[{i}] | Topic- {doc.metadata.get('topic')} |")
    print(f"Content- {doc.page_content}\n")

Query: How is CRISPR acting as a big enabler in creating personalized medicine?
[1] | Topic- medicine |
Content- CRISPR gene editing technology has revolutionized medical genomics, enabling precise modifications to DNA sequences that were previously impossible. Researchers are using genomic data to develop personalized medicine approaches, tailoring treatments based on an individual's genetic profile. Recent breakthroughs in mRNA technology, accelerated by COVID-19 vaccine development, are now being applied to cancer immunotherapy and rare genetic disorders. Hospital information systems are increasingly integrating genomic data to support clinical decision-making at the point of care.

[2] | Topic- quantum_computing |
Content- Quantum computing has reached a critical milestone with several companies demonstrating quantum advantage on specific computational tasks. Error correction remains the biggest challenge, as current quantum processors are highly susceptible to noise and decoherenc

### LLM Based Compressor

In [22]:
compressor = LLMChainExtractor.from_llm(llm)

compressor_retriever = ContextualCompressionRetriever(
    base_compressor= compressor,
    base_retriever= base_retriever
)

In [24]:
print(compressor_retriever)

base_compressor=LLMChainExtractor(llm_chain=PromptTemplate(input_variables=['context', 'question'], input_types={}, output_parser=NoOutputParser(), partial_variables={}, template='Given the following question and context, extract any part of the context *AS IS* that is relevant to answer the question. If none of the context is relevant return NO_OUTPUT.\n\nRemember, *DO NOT* edit the extracted parts of the context.\n\n> Question: {question}\n> Context:\n>>>\n{context}\n>>>\nExtracted relevant parts:')
| ChatOpenAI(output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachme

In [ ]:
compressor_result = compressor_retriever.invoke(query)

for i, doc in enumerate(compressor_result):
    print(f"[{i}] | Metadata- {doc.metadata} |")
    print(doc.page_content)

[0] | Metadata- {'topic': 'medicine'} |
CRISPR gene editing technology has revolutionized medical genomics, enabling precise modifications to DNA sequences that were previously impossible. Researchers are using genomic data to develop personalized medicine approaches, tailoring treatments based on an individual's genetic profile.


### Embedding Based Filtering

In [34]:
query = "How is CRISPR acting as a big enabler in creating personalized medicine?"
embeddings_filter = EmbeddingsFilter(embeddings= embedding, similarity_threshold= 0.54)

compressor_retriever_embedding = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,
    base_retriever=base_retriever
)

In [36]:
compressor_retriever_embedding

ContextualCompressionRetriever(base_compressor=EmbeddingsFilter(embeddings=OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7489fa2387d0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7489fa239090>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True), similarity_fn=<function cosine_similarity at 0x748a1c158680>, k=20, similarity_threshold=0.54), b

In [35]:
final_result_emb = compressor_retriever_embedding.invoke(query)
final_result_emb

[_DocumentWithState(metadata={'topic': 'medicine'}, page_content="CRISPR gene editing technology has revolutionized medical genomics, enabling precise modifications to DNA sequences that were previously impossible. Researchers are using genomic data to develop personalized medicine approaches, tailoring treatments based on an individual's genetic profile. Recent breakthroughs in mRNA technology, accelerated by COVID-19 vaccine development, are now being applied to cancer immunotherapy and rare genetic disorders. Hospital information systems are increasingly integrating genomic data to support clinical decision-making at the point of care.", state={'embedded_doc': [-0.016021728515625, 0.02154541015625, 0.037841796875, 0.043701171875, -0.042755126953125, -0.052734375, 0.017822265625, 0.0181884765625, -0.019378662109375, -0.0197296142578125, 0.0186004638671875, -0.0267486572265625, 0.01557159423828125, 0.0194091796875, 0.00618743896484375, -0.040191650390625, -0.0010242462158203125, -0.00

### DocumentCompressor Pipeline

In [37]:
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[embeddings_filter, compressor]
)

compression_retriever_pipeline = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,
    base_retriever=base_retriever
)

In [38]:
compression_retriever_pipeline

ContextualCompressionRetriever(base_compressor=DocumentCompressorPipeline(transformers=[EmbeddingsFilter(embeddings=OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7489fa2387d0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7489fa239090>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True), similarity_fn=<function cosine_similarity at 0x748a1c158

In [41]:
final_result = compression_retriever_pipeline.invoke(query)
final_result

[Document(metadata={'topic': 'medicine'}, page_content="CRISPR gene editing technology has revolutionized medical genomics, enabling precise modifications to DNA sequences that were previously impossible. Researchers are using genomic data to develop personalized medicine approaches, tailoring treatments based on an individual's genetic profile.")]

In [44]:
print(final_result[0].page_content)

CRISPR gene editing technology has revolutionized medical genomics, enabling precise modifications to DNA sequences that were previously impossible. Researchers are using genomic data to develop personalized medicine approaches, tailoring treatments based on an individual's genetic profile.
